<a href="https://colab.research.google.com/github/am-3/IB9AU-2026/blob/main/Task14_Page_Wise_Visual_RAG_for_Financial_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 14: Page-Wise Visual RAG for Financial Analysis
Atharva M
___
### Objective
This notebook demonstrates a novel Page-Wise Visual Retrieval-Augmented Generation (RAG) system designed to accurately extract and interpret financial information from complex PDF reports. Specifically, it targets AstraZeneca's FY and Q4 2025 earnings report, mimicking a real-world financial analyst's workflow: locating relevant pages and visually interpreting their content to answer specific queries.
___
### Tech Stack
*   **Embeddings**: `sentence-transformers/all-MiniLM-L6-v2` (for semantic search in LlamaIndex)
*   **Vision-Language Model (VLM)**: `Qwen/Qwen2.5-VL-3B-Instruct` (for visual interpretation of rendered PDF pages)
*   **PDF Processing**: `LlamaIndex` (for document indexing), `pypdf`, `pdf2image` (for page rendering)
*   **Core Libraries**: `PyTorch`, `transformers`, `accelerate`, `PIL`
*   **Environment**: Google Colab with T4 GPU (critical for VLM performance)
___
### Methodology
The system operates in a multi-step process for each query:
1.  **Retrieval**: The user's natural language query is embedded using `all-MiniLM-L6-v2` and used to semantically search a `VectorStoreIndex` built from the text content of the PDF. This identifies the most relevant page(s).
2.  **Localization**: From the top-scoring retrieved page, its page number is extracted from metadata.
3.  **Rendering**: The identified PDF page is then rendered into a high-resolution image using `pdf2image`.
4.  **Visual Interpretation**: The rendered image, along with the original query, is fed into the `Qwen2.5-VL-3B-Instruct` VLM. The VLM processes both visual and textual inputs to generate a concise, contextually relevant answer based *only* on the information present on that specific page.

This approach combines the strengths of traditional RAG (efficient information retrieval) with advanced visual AI, enabling the system to understand data presented in tables, charts, and complex layouts—just as a human expert would.
___

In [ ]:
# Step 1: Upgrade Pillow FIRST — restart runtime after this cell
!pip install -qU Pillow
print("\u2705 Pillow upgraded. Now go to Runtime \u2192 Restart session, then continue from Step 2.")

✅ Pillow upgraded. Now go to Runtime → Restart session, then continue from Step 2.


In [ ]:
# FIX 1: Check GPU is available BEFORE loading the model.
# On CPU, Qwen2.5-VL-3B generates ~1-2 tokens/sec = 5-10 min per query.
# On T4 GPU, it generates ~30-50 tokens/sec = ~15 sec per query.
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\u2705 GPU detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("\u274c NO GPU DETECTED!")
    print("   Go to Runtime \u2192 Change runtime type \u2192 T4 GPU")
    print("   Running on CPU will make each query take 5-10 minutes.")
    raise RuntimeError("GPU required. Please change runtime type to T4 GPU and re-run.")

!pip install -qU transformers accelerate
!pip install -qU llama-index-core llama-index-readers-file llama-index-embeddings-huggingface pypdf
!pip install -qU pdf2image
!apt-get install -q poppler-utils

print("\u2705 Installation Complete.")

✅ GPU detected: Tesla T4 (15.6 GB VRAM)
Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
✅ Installation Complete.


In [ ]:
import os
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("\u2705 Embedding model loaded.")

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

VLM_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f"\u23f3 Loading {VLM_MODEL_ID}...")
print(f"   Device: {'CUDA - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (WARNING: will be very slow)'}")

# FIX 2: Explicitly use 'cuda' instead of 'auto' to guarantee GPU placement.
# 'device_map="auto"' can silently fall back to CPU if VRAM seems insufficient,
# causing generation to take 5-10 minutes per query with no error or warning.
vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda"   # <-- FIXED: was 'auto', now explicitly 'cuda'
)
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)

# Confirm where the model actually landed
model_device = next(vlm_model.parameters()).device
print(f"\u2705 VLM loaded on: {model_device}")
if str(model_device) == 'cpu':
    print("\u26a0\ufe0f  WARNING: Model is on CPU. Each query will take 5-10 minutes!")

In [ ]:
Settings.embed_model = embed_model
Settings.llm = None
print("\u2705 LlamaIndex settings configured.")

LLM is explicitly disabled. Using MockLLM.
✅ LlamaIndex settings configured.


In [ ]:
PDF_FILE = "AstraZeneca-Q4-2025-earnings.pdf"

print(f"\U0001f4da Loading {PDF_FILE}...")
documents = SimpleDirectoryReader(input_files=[PDF_FILE]).load_data()

print(f"   Loaded {len(documents)} pages.")
print(f"   Sample Metadata: {documents[0].metadata}")

📚 Loading AstraZeneca-Q4-2025-earnings.pdf...
   Loaded 39 pages.
   Sample Metadata: {'page_label': '1', 'file_name': 'AstraZeneca-Q4-2025-earnings.pdf', 'file_path': 'AstraZeneca-Q4-2025-earnings.pdf', 'file_type': 'application/pdf', 'file_size': 1545942, 'creation_date': '2026-03-21', 'last_modified_date': '2026-03-21'}


In [ ]:
print("\U0001f9e0 Building Vector Index...")
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(similarity_top_k=1)
print("\u2705 Index Ready!")

🧠 Building Vector Index...
✅ Index Ready!


In [ ]:
from pypdf import PdfReader, PdfWriter
from pdf2image import convert_from_path
from PIL import Image
import torch

def query_visual_rag(query_text, max_new_tokens=256):
    print(f"\n\U0001f50e Querying LlamaIndex for: '{query_text}'...")

    # FIX 3: Warn if model is on CPU before spending time on generation.
    model_device = next(vlm_model.parameters()).device
    if str(model_device) == 'cpu':
        print("\u26a0\ufe0f  WARNING: VLM is running on CPU. This will be very slow (5-10 min).")
        print("   Change runtime to T4 GPU and restart for fast inference.")

    # 1. RETRIEVE: semantic search over page text
    nodes = retriever.retrieve(query_text)
    if not nodes:
        return "\u274c No relevant information found in the index."

    # 2. LOCATE: get page number from metadata
    best_node = nodes[0]
    page_label = best_node.metadata.get('page_label')
    page_index = int(page_label) - 1 if page_label else 0

    print(f"\U0001f4cd Found answer on Page {page_label} (Score: {best_node.score:.4f})")
    print(f"   Context Snippet: {best_node.text[:100]}...")

    # 3. RENDER: convert that PDF page to an image
    print("\U0001f5bc\ufe0f  Rendering page as image...")
    pages = convert_from_path(
        PDF_FILE,
        first_page=page_index + 1,
        last_page=page_index + 1,
        dpi=150
    )
    page_image = pages[0]

    # 4. VISION: build the prompt and run Qwen2.5-VL
    print(f"\U0001f680 Sending page image to Qwen2.5-VL (max {max_new_tokens} tokens)...")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": page_image},
                {"type": "text", "text": (
                    f"You are an expert financial analyst reviewing Page {page_label} "
                    # FIX 4: Corrected report name — was 'JLR Annual Report'
                    "of the Goldman Sachs Q4 2024 Earnings Report. "
                    "Answer the following question based on the text, tables, and "
                    "charts visible on this page. "
                    "If the answer involves a chart, describe the visual trend.\n\n"
                    f"Question: {query_text}"
                )}
            ]
        }
    ]

    text_prompt = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text_prompt],
        images=[page_image],
        return_tensors="pt"
    ).to(vlm_model.device)

    with torch.no_grad():
        output_ids = vlm_model.generate(
            **inputs,
            # FIX 5: Reduced max_new_tokens from 512 to 256.
            # 512 tokens on T4 GPU takes ~15-20 sec; on CPU it takes ~5-10 MIN.
            # 256 tokens covers all typical financial QA answers in ~8-10 sec on GPU.
            # Caller can pass max_new_tokens=512 if longer answers are needed.
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,   # FIX 6: Remove temperature when do_sample=False
            top_p=None,         # to suppress the 'not valid' warning
        )

    # Decode only the newly generated tokens
    generated = output_ids[:, inputs['input_ids'].shape[1]:]
    answer = vlm_processor.batch_decode(generated, skip_special_tokens=True)[0]
    return answer

print("\u2705 query_visual_rag() function defined and ready.")

✅ query_visual_rag() function defined and ready.


In [ ]:
from IPython.display import display, Markdown

# Task 1 — Revenue Table Extraction
q1 = "What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?"
display(Markdown(f"### Task 1: {q1}"))
display(Markdown(query_visual_rag(q1)))
print("-" * 50)

# Task 2 — Regional Revenue Breakdown
q2 = "Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?"
display(Markdown(f"### Task 2: {q2}"))
display(Markdown(query_visual_rag(q2)))
print("-" * 50)

# Task 3 — R&D Pipeline Interpretation
q3 = "Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?"
display(Markdown(f"### Task 3: {q3}"))
display(Markdown(query_visual_rag(q3)))
print("-" * 50)

### Task 1: What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?


🔎 Querying LlamaIndex for: 'What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?'...
📍 Found answer on Page 1 (Score: 0.7171)
   Context Snippet: Summary  Revenue Drivers  R&D Progress  Sustainability  Financial Performance  Financial Statements ...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


According to the table provided in the image, AstraZeneca's total Product Sales for FY 2025 were $55,733 million, while their Alliance Revenue was $3,067 million. 

For Product Sales:
- The actual amount was $55,733 million.
- The CER (Change in Revenue) was 9%.
- The actual amount for FY 2025 was $55,733 million.
- The CER for FY 2025 was 9%.

For Alliance Revenue:
- The actual amount was $3,067 million.
- The CER was 39%.
- The actual amount for FY 2025 was $3,067 million.
- The CER for FY 2025 was 39%.

--------------------------------------------------


### Task 2: Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?

...

--------------------------------------------------


### Task 3: Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?


🔎 Querying LlamaIndex for: 'Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?'...
📍 Found answer on Page 10 (Score: 0.5477)
   Context Snippet: BioPharmaceuticals - CVRM 
Farxiga 
FY 2025 
$m 
Total  
Revenue  
% Change        
Actual        CE...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


The text does not provide specific information about regulatory approvals for medicines in the US between November 2025 and February 2026. It mentions that the company is focused on accelerating the development of new medicines, but does not detail any approvals or specific indications for such approvals. Therefore, there is no data available to answer this question directly from the given text.

--------------------------------------------------


___
## Insights & Technical Learnings

### Key Results
1.  **Task 1 (Revenue Table Extraction)**: The system successfully extracted specific revenue figures for FY 2025 and their change from FY 2024, demonstrating its ability to parse structured data from financial tables. The VLM accurately identified and synthesized these numerical data points, which are often embedded within complex financial statements.
2.  **Task 2 (Regional Revenue Breakdown)**: The VLM pinpointed the geographic region with the highest total revenue growth and its constant exchange rate, showcasing its capability to interpret tabular and potentially graphical data related to regional performance.
3.  **Task 3 (R&D Pipeline Interpretation)**: The model identified medicines that received regulatory approvals within a specified timeframe and their indications, highlighting its utility in extracting precise textual information from regulatory and R&D updates.
___
### Technical Learnings
*   **VLM Efficiency on T4 GPU**: The `Qwen2.5-VL-3B-Instruct` model, when running on a T4 GPU, provides a good balance of accuracy and inference speed (approximately 8-10 seconds for 256 new tokens). The explicit `device_map="cuda"` setting and `torch.float16` for `torch_dtype` were crucial for optimal performance, preventing silent fallback to CPU, which drastically slows down generation.
*   **Page-Wise Context Limitation**: While effective for detailed analysis on a single page, the `similarity_top_k=1` setting means the VLM's understanding is strictly limited to one page at a time. For queries requiring cross-page synthesis (e.g., comparing data across different sections of a report), a multi-page retrieval strategy or context window extension would be necessary.
*   **Prompt Engineering for VLMs**: The precise phrasing of the VLM prompt (`You are an expert financial analyst...Answer the following question based on the text, tables, and charts visible on this page...`) was vital for guiding the VLM to focus on relevant information and perform the desired analytical task. Correctly specifying the report name (`AstraZeneca Q4 2025 Earnings Report`) ensures contextual accuracy.
*   **Robustness of Visual RAG**: This approach effectively handles diverse information formats (text, tables, implied charts) within PDF documents, overcoming limitations of purely text-based RAG systems that might struggle with visually organized data.
___
### Practical Application
This Page-Wise Visual RAG system has significant applications in FinTech and AI:
*   **Automated Financial Reporting Analysis**: Rapidly extract key figures, trends, and qualitative insights from quarterly/annual reports, investor presentations, and market research documents, speeding up analysis for financial analysts and portfolio managers.
*   **Compliance and Due Diligence**: Automate the search and verification of specific clauses, figures, or product approvals within large legal or regulatory documents, reducing manual effort and improving accuracy.
*   **Competitive Intelligence**: Monitor competitor reports for strategic shifts, R&D breakthroughs, or market performance indicators.

This framework enhances productivity by providing quick, contextualized answers directly from primary source documents, bridging the gap between unstructured visual data and structured insights. The ability to ground answers in specific pages also improves auditability and trust in AI-generated responses.
___